# 🧠 Phase 4 — Explainability & Analyst Reports (v4)

### 🔍 Context from Previous Phase
In **Phase 3**, we built the *graph features* layer.  
While those network metrics (degree, component size, fraud-neighbor ratio) didn’t boost AUC, they added vital **context** — showing how transactions are linked to other entities.  
This interpretability foundation sets the stage for **Phase 4**, where the goal is to **translate model scores into clear, human explanations.**

---

### 🎯 Objective
- Combine model contributions (LightGBM `pred_contrib`) with graph metrics and simple rules  
  → produce short, analyst-ready reason texts per transaction.  
- Export explanations as `valid_reasons_v4.csv` for dashboards or audits.

---

### 📦 Inputs / Outputs
| Input | Source |
|-------|---------|
| `lgbm_v3.joblib` | best model from Phase 3 |
| `valid_processed.parquet` | processed validation set |
| `graph_features_valid_v3.parquet` | graph metrics per transaction |

| Output | Purpose |
|---------|----------|
| `valid_reasons_v4.csv` | explanations per transaction |
| `reason_summary_top10.csv` | most frequent reasons |


In [3]:
# ============================================================
#  Phase 4 — Explainability & Analyst Reports (v4)
# ============================================================

import pandas as pd, numpy as np, joblib, json, pathlib

ROOT = pathlib.Path("/Users/animeshchoubey/Downloads/Capstone project").resolve()
ART = ROOT / "artifacts"; ART.mkdir(exist_ok=True, parents=True)
PROCESSED = ROOT / "processed"

# Load model and data
clf = joblib.load(ART / "lgbm_v3.joblib")
valid = pd.read_parquet(PROCESSED / "valid_processed.parquet")
graph_feats = pd.read_parquet(ART / "graph_features_valid_v3.parquet")

# Merge graph features into validation data
valid = pd.concat([valid.reset_index(drop=True), graph_feats.reset_index(drop=True)], axis=1)
print("Valid shape:", valid.shape)


Valid shape: (118108, 580)


## 1️⃣ Compute Feature Contributions

LightGBM provides per-feature additive contributions via `predict(..., pred_contrib=True)`.  
These values act like SHAP scores, quantifying how much each feature pushes the fraud probability up or down for every transaction.


In [5]:
contribs = clf.booster_.predict(valid.drop(columns=["isFraud"]), pred_contrib=True)
feat_names = clf.booster_.feature_name()
contrib_df = pd.DataFrame(contribs[:, :-1], columns=feat_names)
print(" Feature contribution matrix shape:", contrib_df.shape)


 Feature contribution matrix shape: (118108, 578)


## 2️⃣ Generate Rule-Based Explanations

We combine:
- **Graph features** → context (“connected to prior frauds”)
- **Transaction amount** → behavioral cues
- **Top model drivers** → most influential features from the contribution matrix



In [7]:
# --- minimal logging helper ---
def ok(label, details=None):
    print(f" {label}" + (f" — {details}" if details else ""))


In [8]:
def generate_reason(row, contrib_row):
    reasons = []

    # --- graph-based rules ---
    if row.get("graph_fraud_ratio_max", 0) > 0.5:
        reasons.append("connected to multiple past frauds")
    if row.get("graph_degree_max", 0) > 20:
        reasons.append("high network connectivity")
    if row.get("graph_comp_size_max", 0) > 1000:
        reasons.append("part of a large entity cluster")

    # --- transaction rule example ---
    if "TransactionAmt" in row and row["TransactionAmt"] > 500:
        reasons.append("unusually high transaction amount")

    # --- top model drivers ---
    top_feats = contrib_row.abs().sort_values(ascending=False).head(2).index.tolist()
    reasons.append("top drivers: " + ", ".join(top_feats))

    return "; ".join(reasons)

reasons_list = [
    generate_reason(valid.iloc[i], contrib_df.iloc[i]) for i in range(len(valid))
]

ok("Generated explanations", f"{len(reasons_list)} rows")


 Generated explanations — 118108 rows


## 3️⃣ Assemble Final Output

Each record now contains:
- `TransactionID`
- `isFraud`
- `fraud_score`
- `reason_text`


In [10]:
# --- FIX DUPLICATE isFraud COLUMNS & BUILD TABLEAU EXPORT ---

import numpy as np
import pandas as pd

# 1) Identify all isFraud columns
isf_idx = [i for i, c in enumerate(valid.columns) if c == "isFraud"]
print("isFraud columns found:", len(isf_idx))

# 2) Keep the first isFraud column as the label
y_series = valid.iloc[:, isf_idx[0]]

# 3) Drop ALL isFraud columns from X so proba works cleanly
X_valid_no_y = valid.drop(columns=[valid.columns[i] for i in isf_idx])

# 4) Assemble final output with synthetic IDs
valid_out = pd.DataFrame({
    "TransactionID": np.arange(len(valid), dtype=np.int64),
    "isFraud": y_series.astype("int8").to_numpy(),
})
valid_out["fraud_score"] = clf.predict_proba(X_valid_no_y)[:, 1]
valid_out["reason_text"] = reasons_list

ok("Assembled final explanations (synthetic IDs)", f"{valid_out.shape[0]} rows")

# 5) Save for Tableau
out_path = ART / "valid_reasons_v4.csv"
valid_out.to_csv(out_path, index=False)
ok("Saved explanations CSV", str(out_path))

# Preview a few rows
valid_out.head(3)


isFraud columns found: 2
 Assembled final explanations (synthetic IDs) — 118108 rows
 Saved explanations CSV — /Users/animeshchoubey/Downloads/Capstone project/artifacts/valid_reasons_v4.csv


,TransactionID,isFraud,fraud_score,reason_text
0,0,1,0.053546,high network connectivity; part of a large ent...
1,1,0,0.045204,high network connectivity; part of a large ent...
2,2,0,0.105640,high network connectivity; part of a large ent...


In [11]:
import re, pandas as pd

# reload in case you restarted the kernel
valid_out = pd.read_csv("/Users/animeshchoubey/Downloads/Capstone project/artifacts/valid_reasons_v4.csv")

# --- A) Top human-readable reasons (ignore "top drivers:" parts) ---
PAT_DRIVER = re.compile(r"^top drivers:\s*", flags=re.IGNORECASE)

frags = (
    valid_out["reason_text"].fillna("").astype(str)
        .str.split("; ")
        .explode()
        .map(lambda s: s.strip())
        .dropna()
        .loc[lambda s: s.ne("")]
)

human_frags = frags[~frags.str.match(PAT_DRIVER)]
top_reasons = human_frags.value_counts().head(10)
print("\nMost common human reasons:\n", top_reasons)

# save for Tableau / report
ART = pathlib.Path("/Users/animeshchoubey/Downloads/Capstone project/artifacts")
top_reasons.to_csv(ART / "reason_summary_top10.csv", header=["count"])
ok("Top reason summary saved", "→ reason_summary_top10.csv")

# --- B) Top driver features ---
driver_lines = frags[frags.str.match(PAT_DRIVER)]
driver_feats = (
    driver_lines.str.replace(PAT_DRIVER, "", regex=True)
                .str.split(",")
                .explode()
                .map(lambda s: s.strip())
                .loc[lambda s: s.ne("")]
)
top_driver_feats = driver_feats.value_counts().head(15)
print("\nTop driver features:\n", top_driver_feats)

top_driver_feats.to_csv(ART / "driver_features_top15.csv", header=["count"])
ok("Top driver features saved", "→ driver_features_top15.csv")



Most common human reasons:
 reason_text
part of a large entity cluster       118108
high network connectivity            118107
unusually high transaction amount      4816
connected to multiple past frauds       231
Name: count, dtype: int64
 Top reason summary saved — → reason_summary_top10.csv

Top driver features:
 reason_text
graph_fraud_ratio_mean    93522
M5__was_missing           27170
graph_fraud_ratio_max     23269
C13                       22211
C1                        10800
C5                         9973
V70                        9192
TransactionAmt             8813
V308                       4909
graph_degree_mean          4130
C14                        3867
D3                         1681
V294                       1677
D15                        1528
V283                       1238
Name: count, dtype: int64
 Top driver features saved — → driver_features_top15.csv


### 🧠 **Business Interpretation — Phase 4 (Explainability & Reports)**

The goal of this phase was to **make model outputs understandable to analysts and business users**.
Each transaction now includes a concise, human-readable *reason text* that links model predictions to clear behavioral or network cues.

**Key outcomes**

* Every validation transaction (118 108 rows) now carries its own explanation.
* Frequent phrases such as **“part of a large entity cluster”** and **“high network connectivity”** reflect the dense nature of the transaction-device-card graph, confirming the model is capturing connection patterns.
* More actionable reasons—**“unusually high transaction amount”** (≈ 4 %) and **“connected to multiple past frauds”** (≈ 0.2 %)—highlight the subset of transactions that deserve analyst review.
* Analysts can now filter or search by these reasons in Tableau to prioritize high-risk clusters.

**Business value**

* **Transparency:** Stakeholders can trace *why* a transaction was flagged instead of relying on opaque scores.
* **Auditability:** Each prediction is accompanied by evidence that can be exported or reported for compliance.
* **Operational impact:** Review teams can group similar explanations (e.g., “amount anomalies” vs. “network anomalies”) to streamline investigations.

> This phase fulfills **North Star Pillar #3 – “Explain Clearly.”**
> The system has moved from *detection* to *communication*—turning model signals into insights people can act on.

---

### ⚙️ **Technical Interpretation — Phase 4 (Explainability Pipeline)**

**Approach**

1. Used LightGBM’s `pred_contrib` to compute per-feature additive contributions (SHAP-like values) for all 118 k validation rows.
2. Combined these numeric contributions with rule-based conditions derived from the graph features:

   * `graph_fraud_ratio_max > 0.5` → “connected to multiple past frauds”
   * `graph_degree_max > 20` → “high network connectivity”
   * `graph_comp_size_max > 1000` → “part of a large entity cluster”
   * `TransactionAmt > 500` → “unusually high transaction amount”
3. Generated reason strings and appended the top 2 model-driver features per record.

**Findings**

* The explanation engine runs entirely locally—no API calls or external LLMs—ensuring **zero additional cost and full reproducibility**.
* Complexity is **O(n × f)** (≈ 118 k × 578 features) → completes in minutes on a standard laptop.
* The top model drivers confirm stability of the underlying LightGBM:
  `graph_fraud_ratio_mean`, `graph_fraud_ratio_max`, `TransactionAmt`, and card-based features dominate contributions.
* Generated CSVs (`valid_reasons_v4.csv`, `reason_summary_top10.csv`, `driver_features_top15.csv`) integrate cleanly with Tableau for visualization.

**Takeaways**

* The pipeline proved deterministic and interpretable; it can be re-run whenever the model or rules update.
* Feature contributions align with business intuition, showing the model leverages both **network context** and **transaction behavior**.
* This step bridges machine output and human understanding, preparing for Phase 5 visualization.

